<a href="https://colab.research.google.com/github/tsekatm/aws-python-data-engineering-challenge/blob/main/multi_projects_cloud_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Multi‑Cloud Data Notebook: 5 Practical Projects
**Generated:** July 08, 2025

This notebook contains five *independent* mini‑projects that showcase how you can mix **AWS** and other public APIs to build useful data products. Each project is self‑contained—feel free to run them in any order once the prerequisites for that project are in place.

## Projects
1. **Video‑to‑Lesson & Quiz** – YouTube → AWS Transcribe → Amazon Bedrock for summary & quiz generation
2. **COVID‑19 Heatmap (ZA)** – Aggregated provincial data visualised with GeoPandas + Folium, hosted on S3
3. **Apple Music → Spotify Migrator** – Serverless pipeline using MusicKit & Spotify Web API orchestrated by AWS Step Functions
4. **Live Sentiment Dashboard** – Twitter stream → AWS Kinesis Data Streams → AWS Comprehend → live Plotly graph
5. **Smart Image Catalogue** – AWS Rekognition labels + Amazon Bedrock alt‑text → indexed in OpenSearch

---

### How to use this notebook
1. **Clone/Download** and install the libraries listed below.
2. Export the environment variables requested in each project’s *Setup* cell (API keys, AWS credentials, etc.).
3. Run the cells for the project you want to explore.
4. Clean up cloud resources when you’re done—**some services incur cost when left running.**


## Global Setup

In [5]:
# Uncomment if running in a fresh environment
!pip install --quiet boto3 botocore pandas geopandas folium plotly pillow
#     yt-dlp python-dotenv openai nbformat requests spotipy tweepy

import os, json, boto3, time
import pandas as pd
from google.colab import userdata # Import userdata

# Load AWS credentials from Colab Secrets Manager
os.environ['AWS_ACCESS_KEY_ID'] = userdata.get('AWS_ACCESS_KEY_ID')
os.environ['AWS_SECRET_ACCESS_KEY'] = userdata.get('AWS_SECRET_ACCESS_KEY')

AWS_REGION = os.getenv("AWS_REGION", "us-east-1")
session = boto3.Session(region_name=AWS_REGION)

print(f"Using AWS region: {AWS_REGION}")

Using AWS region: us-east-1


## 1  · Video‑to‑Lesson & Quiz (YouTube + AWS Transcribe + Amazon Bedrock)

**Goal:** Turn any public YouTube video into a short lesson with an auto‑generated quiz.

**Workflow:**
1. Download video audio using `yt‑dlp`.
2. Upload the audio file to an S3 bucket (`video‑lessons‑<your‑suffix>`).
3. Start an **AWS Transcribe** job → returns a JSON transcript.
4. Use **Amazon Bedrock** (or OpenAI) to summarise the transcript into lesson notes **and** generate MCQ quiz items.
5. Display the lesson & quiz inline in the notebook.

> **Cost tips:** Transcribe is priced per audio‑second; Bedrock LLM calls are per token. Delete the S3 object afterward.


In [8]:
# 📦 Setup and imports
!pip -q install yt-dlp boto3 botocore

import os, time, json, requests
import boto3
from botocore.exceptions import ClientError
from IPython.display import Markdown

# 🪪 AWS Credentials (Colab-safe)
# Make sure these are set using %env or IAM role if in SageMaker
session = boto3.Session(region_name="us-east-1")

# ---- Config ----
YOUTUBE_URL = "https://www.youtube.com/watch?v=hrvx8Nv9eQA&list=PLJq-63ZRPdBsPWE24vdpmgeRFMRQyjvvj"  # <-- replace
S3_BUCKET = "video-lessons-demo"
S3_KEY = "downloaded_audio.mp3"
TRANSCRIBE_JOB_NAME = "demo-" + str(int(time.time()))

# ---- 1. Download audio ----
if not os.path.exists("downloaded_audio.mp3"):
    print("🎧 Downloading audio from YouTube...")
    !yt-dlp -x --audio-format mp3 -o "downloaded_audio.%(ext)s" {YOUTUBE_URL}
else:
    print("✅ Audio already downloaded")

# ---- 2. Upload to S3 ----
s3 = session.client("s3")

# Create bucket only if it doesn't exist
try:
    s3.head_bucket(Bucket=S3_BUCKET)
    print(f"✅ Bucket already exists: {S3_BUCKET}")
except ClientError as e:
    code = e.response["Error"]["Code"]
    if code in ("404", "NoSuchBucket", "NotFound"):
        print(f"📦 Creating S3 bucket: {S3_BUCKET}")
        s3.create_bucket(Bucket=S3_BUCKET)
    elif code == "403":
        raise RuntimeError(f"🚫 Bucket name '{S3_BUCKET}' already taken or access denied.")
    else:
        raise

# Upload the file
print("📤 Uploading to S3...")
s3.upload_file("downloaded_audio.mp3", S3_BUCKET, S3_KEY)

# ---- 3. Start Transcribe job ----
transcribe = session.client("transcribe")

media_uri = f"s3://{S3_BUCKET}/{S3_KEY}"

try:
    print("📝 Starting Transcribe job...")
    response = transcribe.start_transcription_job(
        TranscriptionJobName=TRANSCRIBE_JOB_NAME,
        Media={"MediaFileUri": media_uri},
        MediaFormat="mp3",
        LanguageCode="en-US"
    )
except ClientError as e:
    if e.response["Error"]["Code"] == "ConflictException":
        print("⚠️ Transcription job already exists. Using existing job.")
    else:
        raise

# ---- 4. Poll until complete ----
print("⏳ Waiting for transcription to finish...")
while True:
    result = transcribe.get_transcription_job(TranscriptionJobName=TRANSCRIBE_JOB_NAME)
    status = result["TranscriptionJob"]["TranscriptionJobStatus"]
    print(f"📡 Status: {status}")
    if status in ("COMPLETED", "FAILED"):
        break
    time.sleep(15)

if status == "FAILED":
    raise RuntimeError("❌ Transcription job failed.")

# ---- 5. Extract transcript ----
transcript_uri = result["TranscriptionJob"]["Transcript"]["TranscriptFileUri"]
transcript_text = requests.get(transcript_uri).json()["results"]["transcripts"][0]["transcript"]

print("📝 Transcript ready.")

# ---- 6. Summarise and quiz via Bedrock (optional) ----
# Requires Bedrock access and ai21.j2-ultra to be enabled in your account
# Uncomment this section if applicable

# bedrock = session.client("bedrock-runtime")
# prompt = f"Summarise the following transcript into key learning points, then create a 5‑question multiple‑choice quiz.\n\n{transcript_text}"
# response = bedrock.invoke_model(
#     body=json.dumps({"prompt": prompt, "max_tokens": 1024}),
#     modelId="ai21.j2-ultra",
#     contentType="application/json",
#     accept="application/json"
# )
# summary = json.loads(response["body"].read())["completions"][0]["data"]["text"]

# ---- 7. Display ----
# Markdown(summary)  # Uncomment after using Bedrock
Markdown(transcript_text[:1000])  # Show preview of transcript


🎧 Downloading audio from YouTube...
[youtube] Extracting URL: https://www.youtube.com/watch?v=hrvx8Nv9eQA
[youtube] hrvx8Nv9eQA: Downloading webpage
[youtube] hrvx8Nv9eQA: Downloading tv client config
[youtube] hrvx8Nv9eQA: Downloading player dbb35e0d-main
[youtube] hrvx8Nv9eQA: Downloading tv player API JSON
[youtube] hrvx8Nv9eQA: Downloading ios player API JSON
[youtube] hrvx8Nv9eQA: Downloading m3u8 information
[info] hrvx8Nv9eQA: Downloading 1 format(s): 251
[download] Destination: downloaded_audio.webm
[download] 100% of    8.09MiB in 00:00:00 at 19.73MiB/s
[ExtractAudio] Destination: downloaded_audio.mp3
Deleting original file downloaded_audio.webm (pass -k to keep)
✅ Bucket already exists: video-lessons-demo
📤 Uploading to S3...
📝 Starting Transcribe job...
⏳ Waiting for transcription to finish...
📡 Status: IN_PROGRESS
📡 Status: IN_PROGRESS
📡 Status: IN_PROGRESS
📡 Status: IN_PROGRESS
📡 Status: COMPLETED
📝 Transcript ready.


The event-driven architecture or EDA pattern is taking center stage in modern software design. With the rise of microservices, big data, and real-time processing. Companies need a scalable and flexible way to handle interactions between different components. In today's video, we'll break down the event-driven architecture pattern, explore why it is gaining popularity, and dive into real world case studies of companies like Netflix and Uber who are leading the way in using this architecture to handle billions of events daily. Let's jump right in. As your application expands and new services are introduced, the traditional request response model becomes less efficient. In a simple case, Service A requests data from Service B, which processes the request and sends a response. However, as more services are introduced, managing these interactions become exponentially complex. Consider this for every interaction, you need to define a request and response. When there are only few services, th

## 2  · COVID‑19 Heatmap over South African Provinces (GeoPandas + Folium)

**Goal:** Create an interactive choropleth of cumulative COVID‑19 cases per province and host the HTML on S3 for public access.

**Workflow:**
1. Download or load a CSV with daily SA COVID stats (e.g. NICD/JHU).
2. Aggregate to latest total per province.
3. Join with provincial boundaries shapefile.
4. Render an interactive **Folium** map inside the notebook.
5. Save to `covid‑za.html` and upload to S3 (`public-read`) so it’s shareable.


In [9]:
import geopandas as gpd
import folium

# ---- 1. Load data ----
# covid_df = pd.read_csv("za_covid_by_province.csv")
# provinces_geo = gpd.read_file("zaf_adm1.geojson")

# ---- 2. Aggregate ----
# latest = covid_df.sort_values("date").groupby("province").last().reset_index()

# ---- 3. Merge ----
# merged = provinces_geo.merge(latest, left_on="NAME_1", right_on="province")

# ---- 4. Plot ----
# m = folium.Map(location=[-28.3, 24.7], zoom_start=5, tiles="CartoDB positron")
# folium.Choropleth(
#     geo_data=merged,
#     data=merged,
#     columns=["province", "cases"],
#     key_on="feature.properties.NAME_1",
#     legend_name="COVID‑19 cases",
#     fill_opacity=0.7,
#     line_opacity=0.1,
# ).add_to(m)

# m


## 3  · Apple Music → Spotify Playlist Migrator (Step Functions)

**Goal:** Move your curated Apple Music playlist to Spotify while preserving track order and avoiding duplicates.

**Architecture:**
```
Notebook (trigger)  →  AWS Step Functions  →  λ • Fetch Apple playlist
                                         ↘  λ • Search & add on Spotify
```
**Steps in notebook:**
1. Collect the MusicKit developer token & user token, and Spotify OAuth token.
2. Kick off the Step Functions execution with the playlist ID.
3. Poll execution status; show a nice progress bar.
4. Display a DataFrame comparing Apple vs. Spotify track URIs (helpful for any misses).

> *Prereq:* Create the two Lambda functions & SFn state‑machine (provided in the `lambda/` folder of this repo).


In [11]:
import boto3, uuid, pandas as pd, json
step = session.client("stepfunctions")

APPLE_PLAYLIST_ID = "pl.u-leyl0kGcjV2Gprr"
EXECUTION_ARN = step.start_execution(
    stateMachineArn="arn:aws:states:REGION:ACCT:stateMachine:PlaylistMigrator",
    name="exec-" + str(uuid.uuid4()),
    input=json.dumps({"applePlaylistId": APPLE_PLAYLIST_ID})
)["executionArn"]

print("Started:", EXECUTION_ARN)

# ---- Poll ----
# while True:
#     desc = step.describe_execution(executionArn=EXECUTION_ARN)
#     if desc["status"] in ("SUCCEEDED", "FAILED", "TIMED_OUT"):
#         break
#     time.sleep(5)

# result = json.loads(desc["output"])
# df = pd.DataFrame(result["mappings"])
# df.head()


InvalidArn: An error occurred (InvalidArn) when calling the StartExecution operation: Invalid Arn: 'Expected the ARN arn:aws:states:REGION:ACCT:stateMachine:PlaylistMigrator to be within region (us-east-1).'

## 4  · Live Twitter Sentiment Dashboard (Kinesis + Comprehend)

**Goal:** Stream tweets containing a keyword (e.g. *Load‑Shedding*) into **AWS Kinesis Data Streams**, classify sentiment with **AWS Comprehend**, and plot a live, updating graph in the notebook.

**Notebook actions:**
1. Start (or confirm) the Kinesis stream `tweets-stream`.
2. Use Tweepy to consume the Twitter filtered stream and push raw JSON into Kinesis (runs in background thread).
3. A second thread reads from the stream, calls `comprehend.detect_sentiment`, and appends to an in‑memory list.
4. Update a **Plotly** line chart every 5 s showing counts of Positive / Neutral / Negative.

> *Hint:* You can optionally deploy the producers/consumers as Lambda functions for 24/7 operation.


In [ ]:
import threading, collections, plotly.graph_objs as go
from IPython.display import clear_output, display

# ---- Skeleton only; fill with your keys ----
# STREAM_NAME = "tweets-stream"
# kinesis = session.client("kinesis")
# comprehend = session.client("comprehend")

sentiment_counts = collections.Counter()

def consumer_loop():
    # continuously poll Kinesis & classify
    pass  # TODO

def plot_loop():
    fig = go.FigureWidget()
    display(fig)
    while True:
        clear_output(wait=True)
        fig.data = []
        fig.add_scatter(x=list(sentiment_counts.keys()),
                        y=list(sentiment_counts.values()),
                        mode="lines+markers")
        display(fig)
        time.sleep(5)

# threading.Thread(target=consumer_loop, daemon=True).start()
# threading.Thread(target=plot_loop, daemon=True).start()


## 5  · Smart Image Catalogue with Auto Alt‑Text (Rekognition + Bedrock)

**Goal:** Catalogue images by detected objects and generate accessible alt‑text for each image.

**Workflow:**
1. Upload a folder of JPEG images to S3 (`image‑catalog‑raw`).
2. Iterate over the objects, calling `rekognition.detect_labels` on each.
3. Feed the top labels into Amazon Bedrock to generate a descriptive alt‑text sentence.
4. Write `{key}.json` with labels + alt‑text back to S3; optionally index into Amazon OpenSearch.
5. Build a simple search function inside the notebook that queries OpenSearch by label.


In [ ]:
import io, json, base64
rek = session.client("rekognition")
bedrock = session.client(service_name="bedrock-runtime")
S3_BUCKET_IMG = "image-catalog-raw"

def process_image(obj_key):
    # ---- 1. Download S3 object into memory ----
    s3obj = s3.get_object(Bucket=S3_BUCKET_IMG, Key=obj_key)["Body"].read()
    # ---- 2. Rekognition ----
    labels = rek.detect_labels(Image={'Bytes': s3obj}, MaxLabels=10)["Labels"]
    label_names = [l["Name"] for l in labels]
    # ---- 3. Bedrock alt‑text ----
    prompt = f"Write an alt‑text for an image that contains: {', '.join(label_names)}."
    alt_text = bedrock.invoke_model(
        body=json.dumps({"prompt": prompt, "max_tokens": 60}),
        modelId="ai21.j2-mid"
    ).decode()
    # ---- 4. Persist ----
    meta = {"labels": label_names, "alt_text": alt_text}
    s3.put_object(Bucket=S3_BUCKET_IMG, Key=obj_key + ".json",
                  Body=json.dumps(meta).encode("utf-8"))
    return meta

# Example:
# result = process_image("sample.jpg")
# result


---
## Clean‑Up

In [ ]:
print("Don't forget to: \n"
      "• Stop any running Transcribe jobs\n"
      "• Delete objects in S3 buckets created by this notebook\n"
      "• Delete the Kinesis stream if not needed\n"
      "• Delete the Step Functions execution history\n"
      "• Shut down any notebook kernels to avoid surprise costs!")